# 02 - Data Quality: Missing Values & Distributions

**Goal for today:** solve the `TotalCharges` mystery from last session, properly check
every column for missing values, and look at how our key numeric features are
distributed.

**Why this matters for MLA-C01 (Domain 1 - Data Preparation for ML):**
The exam tests your ability to identify data quality issues that aren't obvious at
first glance - a column silently containing blanks disguised as text is a classic
example. On AWS, this kind of check is exactly what you'd configure a
**SageMaker Data Wrangler "Data Quality and Insights Report"** to catch automatically,
but you need to understand what it's actually looking for underneath.

## Step 0: Reload the data

Each notebook session starts fresh - the kernel doesn't remember `df` from last time.

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/telco_churn.csv')
df.shape

(7043, 21)

## Step 1: The `TotalCharges` mystery - part 1

**What this cell does:** `df.isnull()` checks every single cell in the DataFrame and
returns `True`/`False` for whether it's a genuine missing value (`NaN`). `.sum()` then
adds those up per column, giving you a missing-value count for each one.

**Prediction before you run this:** based on last session, do you expect `TotalCharges`
to show any missing values here? Run it and see if you're right.

In [2]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

If you predicted zero, you were right - and that's the trap. `TotalCharges` shows
**0 missing values**, but we already know it's stored as `object` (text) instead of a
number. `.isnull()` only catches genuine `NaN` values. It can't catch something that
*looks* like a normal string but is actually empty or just whitespace - pandas has no
idea that's "missing," because as far as it's concerned, `' '` is a perfectly valid
string value.

This is exactly why checking `.isnull().sum()` alone is not enough - it's a real
gotcha the exam likes to test, because it's easy to declare a dataset "clean" after
running one check that gives all zeros.

## Step 2: The `TotalCharges` mystery - part 2

**What this cell does:** `.str.strip()` removes leading/trailing whitespace from every
value in the column, then we compare the result to an empty string `''`. This creates a
boolean mask - `True` for every row where `TotalCharges` is blank (or just spaces) once
stripped. Wrapping it in `df[...]` filters the DataFrame down to only those rows.

In [3]:
blank_mask = df['TotalCharges'].str.strip() == ''
df[blank_mask][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']]

,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


There they are - 11 rows hiding in plain sight. Now look at the `tenure` column for
these specific rows. Notice anything?

Every single one of them has `tenure == 0` - these are brand new customers who
signed up but haven't been billed a full cycle yet. `TotalCharges` being blank isn't
random corruption - it's logically consistent with a customer having zero billing
history. That's an important distinction: **missing data can be a genuine data entry
error, or it can be a legitimate real-world state that just needs correct handling**.
For these rows, `TotalCharges` should logically be `0`, not missing.

## Step 3: Fix `TotalCharges`

**What this cell does:** `pd.to_numeric()` converts a column to a numeric type.
The `errors='coerce'` argument tells pandas: "if you hit something you can't convert
to a number (like our blank strings), don't crash - just turn it into `NaN` instead."
This turns our hidden blanks into *proper*, detectable missing values.

In [4]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].dtype

dtype('float64')

In [5]:
df['TotalCharges'].isnull().sum()

np.int64(11)

Now `.isnull().sum()` correctly reports 11 - because we converted the sneaky blank
strings into real `NaN` values. This is the whole point of today's lesson: **the
dtype of a column tells you almost as much as the values themselves.**

We won't decide how to fill these 11 values yet (that's a Domain 1 topic called
*imputation*, coming in a later session) - today is about detection, not repair.

## Step 4: Check every column properly, now that types are fixed

In [6]:
df.isnull().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

With `TotalCharges` now correctly typed, this check is trustworthy. You should see
`TotalCharges` show 11 missing, and everything else show 0 - meaning this dataset is
otherwise clean of genuine missing values. Not every dataset gets off this easy.

## Step 5: Look at how key numeric features are distributed

**What this cell does:** `.describe()` is a method that computes summary statistics -
count, mean, standard deviation, min, max, and the 25th/50th/75th percentiles - for
every numeric column in one call. This is your fastest way to get a feel for a
column's shape without plotting anything yet.

In [7]:
df[['tenure', 'MonthlyCharges', 'TotalCharges']].describe()

,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7032.000000
mean,32.371149,64.761692,2283.300441
std,24.559481,30.090047,2266.771362
min,0.000000,18.250000,18.800000
25%,9.000000,35.500000,401.450000
50%,29.000000,70.350000,1397.475000
75%,55.000000,89.850000,3794.737500
max,72.000000,118.750000,8684.800000


A few things worth reading out of this table:
- **`tenure`** ranges from 0 to some max (in months) - check the `max` value against
  what you'd expect for "years as a customer"
- **`MonthlyCharges`** gives you a feel for the price range Telco customers pay
- **`TotalCharges`** now correctly excludes the 11 blank rows from its calculations
  (count will be 7043 - 11 = 7032)

Compare the `mean` and `50%` (median) for each column - if they're very different,
that's a hint the column is skewed rather than evenly spread, which matters for some
modeling choices later.

---

**That's it for today.** Small, complete increment:
- solved the `TotalCharges` mystery: 11 blank strings hiding from `.isnull()`, all
  belonging to brand-new customers with `tenure == 0`
- converted `TotalCharges` to a proper numeric type with `pd.to_numeric(..., errors='coerce')`
- ran a trustworthy missing-value check across the whole dataset
- got first summary statistics on the three numeric features

**Next session:** we'll visualize these distributions with plots, look at churn rate
broken down by category (e.g. does `Contract` type affect churn?), and start forming
hypotheses about which features matter most before we touch a model.